In [0]:

import requests  # Faz requisições HTTP para consultar as APIs de CEP
import json      # Converte dados entre JSON e objetos Python (dict)
import re        # Usa expressões regulares para limpar e validar o CEP
import time      # Controla pausas entre tentativas (retry e rate limiting)

from datetime import datetime, timezone  # Registra data e hora da consulta em UTC
from pyspark.sql.types import *           # Define os tipos das colunas do DataFrame Spark

# CONFIGURAÇÃO
# cep = "08717310"  # Altere este CEP para testar
dbutils.widgets.text("cep", "08717310", "CEP para consulta")
cep = dbutils.widgets.get("cep")
TABELA = "geocep.default.ref_brasilapi"

# Função que recebe um CEP como texto (str) e retorna um dicionário (dict)
def consultar_cep(cep: str) -> dict:

    # Converte o CEP para texto e remove tudo que não for número (hífen, ponto, espaço etc.)
    cep = re.sub(r"\D", "", str(cep))

    if len(cep) != 8:
        return {"cep": cep, "status": "CEP_INVALIDO"}

    # 2. Primeiro procura no cache dos últimos 6 meses
    cache = spark.sql(f"""
        SELECT *
        FROM {TABELA}
        WHERE cep = '{cep}'
          AND status = 'OK'
          AND data_consulta >= add_months(current_timestamp(), -6)
        ORDER BY data_consulta DESC
        LIMIT 1
    """).collect()

    if cache:
        print("CEP encontrado no CACHE!")
        return cache[0].asDict()

    # 3. Se não encontrou, consulta a API
    print("CEP ausente ou vencido. Consultando BrasilAPI...")

    url = f"https://brasilapi.com.br/api/cep/v2/{cep}"

    for tentativa in range(1, 4):
        try:
            resposta = requests.get(url, timeout=10)

            # Na BrasilAPI, CEP inexistente retorna HTTP 404
            if resposta.status_code == 404:
                return {"cep": cep, "status": "NAO_ENCONTRADO"}

            if resposta.status_code in (429, 500, 502, 503, 504):
                if tentativa < 3:
                    time.sleep(2 ** tentativa)
                    continue

            resposta.raise_for_status()
            dados = resposta.json()

            registro = {
                "cep": cep,
                "fonte": "brasilapi",
                "data_consulta": datetime.now(timezone.utc),
                "status": "OK",
                "http_status": resposta.status_code,
                "tentativas": tentativa,
                "json_original": json.dumps(
                    dados, ensure_ascii=False
                )
            }

            # 4. Gravar ou atualizar o CEP na tabela Delta
            schema = StructType([
                StructField("cep", StringType()),
                StructField("fonte", StringType()),
                StructField("data_consulta", TimestampType()),
                StructField("status", StringType()),
                StructField("http_status", IntegerType()),
                StructField("tentativas", IntegerType()),
                StructField("json_original", StringType())
            ])

            df = spark.createDataFrame([registro], schema)
            df.createOrReplaceTempView("novo_cep_brasilapi")

            spark.sql(f"""
                MERGE INTO {TABELA} AS destino
                USING novo_cep_brasilapi AS origem
                ON destino.cep = origem.cep
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)

            print("CEP consultado e salvo na tabela!")
            return registro

        except (requests.RequestException, ValueError) as erro:
            print(f"Tentativa {tentativa}: {erro}")

            if tentativa < 3:
                time.sleep(2 ** tentativa)

    return {"cep": cep, "status": "ERRO_API"}


# EXECUTAR
resultado = consultar_cep(cep)
print(json.dumps(resultado, indent=2, ensure_ascii=False, default=str))
